# HC 6: First Look at the Data
## CSci 127 · Honors Section · Fall 2026

In this in-lab session you will build the core dataset your team will use for the rest of the semester.

By the end of this notebook you will have:
- Loaded the two raw data files into pandas DataFrames
- Merged them so that every Census tract has both mobility outcomes **and** geographic coordinates
- Filtered the merged data down to New York City's five boroughs
- Saved a clean `nyc_opportunity.csv` file your team will reuse in HC 7 through HC 11
- Made your first plots of this dataset

## Before you start: save your own copy

This notebook opens read-only. Before you run anything, go to **File → Save a copy in
Drive**. That gives you your own copy in Google Drive that you can edit and that saves your
work as you go. Do all of your work in that copy.

You do not need to install anything. Colab already has Python, pandas, and matplotlib ready
to use.


## Imports

We start, as always, by importing the libraries we will need.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

---
## Part 1: Loading the Two Data Files

### File 1 — Opportunity Atlas outcomes (`tract_outcomes_simple.csv`)

This file contains the Opportunity Atlas data for every Census tract in the United States.
Every row is one tract; the columns are the outcome variables measured by the researchers.

Let's load it and take a first look.

In [ ]:
# We load the data directly from the class GitHub repo so this notebook runs in
# Colab with no file upload. The file is gzipped (about 15 MB); pandas reads .gz
# straight from the URL and unzips it for you. This cell may take 10-30 seconds.
outcomes = pd.read_csv('https://raw.githubusercontent.com/HunterCSci127/HunterCSci127.github.io/master/files/tract_outcomes_simple.csv.gz')
print("Shape (rows, columns):", outcomes.shape)
outcomes.head(3)

That's a large file — it covers Census tracts in all 50 states.

Let's see which columns are in it.

In [ ]:
print("Columns in outcomes file:")
for col in outcomes.columns:
    print(" ", col)

The first three columns, `state`, `county`, and `tract`, are the geographic identifiers.
They tell us *where* each row is.

The remaining columns are outcome variables. Our main variable this semester is `kfr_pooled_pooled_p25`.

### File 2 — Census tract centroids (`CenPop2010_Mean_TR36.txt`)

The Atlas file has no coordinates, just state/county/tract codes.
To plot tracts on a map, we need their latitude and longitude.
This second file provides the geographic center (centroid) of every Census tract in New York State.

In [ ]:
# Loaded directly from the class GitHub repo (small file, loads quickly).
centroids = pd.read_csv('https://raw.githubusercontent.com/HunterCSci127/HunterCSci127.github.io/master/files/CenPop2010_Mean_TR36.txt')
print("Shape:", centroids.shape)
centroids.head(3)

Each row here is one Census tract in New York State, identified by `STATEFP` (state code),
`COUNTYFP` (county code), and `TRACTCE` (tract number).
The `LATITUDE` and `LONGITUDE` columns give the geographic center of that tract.

---
## Part 2: Merging the Two DataFrames

Right now we have two separate tables:
- `outcomes` has mobility data but **no coordinates**
- `centroids` has coordinates but **no mobility data**

We want one table that has both. We do this with a **merge**: we match rows from the two tables
that describe the same Census tract, and combine their columns.

A tract is only fully identified by its **state, county, and tract** codes together. County codes
are reused across states (there is a county 5 in many states, not just New York), so matching on
county and tract alone would wrongly pair a New York tract with same-numbered tracts in other
states. We merge on all three keys so each row matches exactly one tract.


In [ ]:
merged = outcomes.merge(
    centroids,
    left_on=['state', 'county', 'tract'],       # column names in the outcomes file
    right_on=['STATEFP', 'COUNTYFP', 'TRACTCE'], # matching column names in the centroids file
    how='inner'                                  # keep only rows that appear in BOTH files
)

print(f"Rows in outcomes file (all US tracts): {len(outcomes):,}")
print(f"Rows in centroids file (NY tracts):    {len(centroids):,}")
print(f"Rows after merging:                    {len(merged):,}")

After merging, every row has both the mobility outcome AND the latitude/longitude.
Let's confirm by looking at a few rows.

In [ ]:
key_cols = ['county', 'tract', 'kfr_pooled_pooled_p25', 'LATITUDE', 'LONGITUDE']
merged[key_cols].head(5)

---
## Part 3: Filtering to New York City

The merged file contains all of New York State. We want only the five NYC boroughs.
Each borough corresponds to one county, identified by a numeric FIPS code:

| County FIPS | Borough       |
|-------------|---------------|
| 5           | Bronx         |
| 47          | Brooklyn      |
| 61          | Manhattan     |
| 81          | Queens        |
| 85          | Staten Island |

We filter the DataFrame to keep only rows where `county` is one of these five values.
We also add a human-readable `borough` column so our plots are easier to read.

In [ ]:
NYC_FIPS = [5, 47, 61, 81, 85]
BOROUGH_NAMES = {
    5:  'Bronx',
    47: 'Brooklyn',
    61: 'Manhattan',
    81: 'Queens',
    85: 'Staten Island'
}

nyc = merged[merged['county'].isin(NYC_FIPS)].copy()
nyc['borough'] = nyc['county'].map(BOROUGH_NAMES)

print(f"NYC tracts in dataset: {len(nyc)}")
nyc[['county', 'borough', 'tract', 'kfr_pooled_pooled_p25', 'LATITUDE', 'LONGITUDE']].head(5)

In [ ]:
# Save the merged NYC dataset. In Colab, to_csv writes to a temporary folder that
# is erased when your session ends, so we also download a copy to your computer.
nyc.to_csv('nyc_opportunity.csv', index=False)

from google.colab import files
files.download('nyc_opportunity.csv')   # this file lands in your computer's Downloads folder

print("Saved to this session and downloaded to your computer: nyc_opportunity.csv")
# Keep this file! You will use it in HC 7 through HC 10.

---
## Part 4: Inspecting the NYC Dataset

### How many Census tracts does each borough have?

In [ ]:
tract_counts = nyc.groupby('borough').size().sort_values(ascending=False)
print("Census tracts per borough:")
print(tract_counts.to_string())

In [ ]:
# Visualize as a bar chart
COLORS = {
    'Bronx':         'tomato',
    'Brooklyn':      'steelblue',
    'Manhattan':     'goldenrod',
    'Queens':        'mediumseagreen',
    'Staten Island': 'mediumpurple'
}

# A quick note on matplotlib before our first plot.
# The FIGURE (fig) is the whole canvas, the outer window the drawing sits in.
# An AXES (ax) is one plot area on that canvas, with its own x-axis, y-axis,
# title, and data. We do our drawing on the axes (ax.bar, ax.set_title, ...),
# not on the figure itself.
#
# plt.subplots() builds the figure and its axes together and hands them back
# as a tuple, which we unpack in one line into fig and ax. We are only making
# a single plot here, but subplots() is still the clean, standard way to get a
# figure and an axes to work with. Called with no numbers it gives exactly one
# axes (later we pass numbers, like subplots(1, 2), to get a grid of several).
# figsize is the width and height of the figure in inches.
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tract_counts.index,
              tract_counts.values,
              color=[COLORS[b] for b in tract_counts.index],
              edgecolor='white')

# Add count labels on top of each bar
for bar, val in zip(bars, tract_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Borough', fontsize=12)
ax.set_ylabel('Number of Census Tracts', fontsize=12)
ax.set_title('Number of Census Tracts per Borough', fontsize=14)
ax.set_ylim(0, tract_counts.max() * 1.12)
plt.tight_layout()
plt.savefig('hc06_tracts_per_borough.png', dpi=150)
plt.show()

### Are there missing values?

Not every tract has a recorded value for `kfr_pooled_pooled_p25`.
When the number of children in a tract is too small to produce a reliable estimate,
the Atlas suppresses the value to protect privacy, so it appears as `NaN` (Not a Number).

Let's check how many tracts are affected in each borough.

In [ ]:
missing = nyc['kfr_pooled_pooled_p25'].isna().groupby(nyc['borough']).sum()
total   = nyc.groupby('borough').size()
pct     = (missing / total * 100).round(1)

summary = pd.DataFrame({
    'total tracts': total,
    'missing':      missing.astype(int),
    'missing %':    pct
})
print("Missing values for kfr_pooled_pooled_p25 by borough:")
print(summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(pct.index, pct.values,
       color=[COLORS[b] for b in pct.index], edgecolor='white')
ax.set_xlabel('Borough', fontsize=12)
ax.set_ylabel('Tracts with Missing Value (%)', fontsize=12)
ax.set_title('Percentage of Tracts Missing the Outcome Variable', fontsize=13)
ax.set_ylim(0, max(pct.values) * 1.3)

for i, (boro, val) in enumerate(pct.items()):
    ax.text(i, val + 0.3, f'{val}%', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

When tracts are missing data, they won't appear in our plots or affect our statistics.
Keep this in mind as you analyze your borough, as some neighborhoods may have more gaps than others.

---
## Part 5: Exploring the Outcome Variable

### What does `kfr_pooled_pooled_p25` mean again?

Breaking down the name:
- **kfr** — kids' family income rank (the outcome we measure)
- **pooled** (twice) — averaged across all genders and all racial groups
- **p25** — for children whose parents were near the **25th percentile** of income (low-income families)

> *In plain English: for children from low-income families who grew up in a Census tract,
> what was their average income rank at age 35?*

The value is between 0 and 1.  A value of **0.40** means those children ended up
at the **40th percentile** of national income on average.  Higher = better.

### Summary statistics across all NYC

In [ ]:
print("Summary statistics for kfr_pooled_pooled_p25 — all NYC tracts:")
nyc['kfr_pooled_pooled_p25'].describe().round(4)

A **histogram** groups values into equal-width bins and shows how many observations fall
into each bin. It gives us a picture of the *shape* of the distribution:
Is it symmetric? Skewed toward one end? Are there tracts with unusually high or low values?

In [ ]:
nyc_mean = nyc['kfr_pooled_pooled_p25'].mean()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(nyc['kfr_pooled_pooled_p25'].dropna(), bins=40,
        color='steelblue', edgecolor='white', alpha=0.85)

ax.axvline(nyc_mean, color='tomato', linewidth=2.5, linestyle='--',
           label=f'NYC mean: {nyc_mean:.3f}')

ax.set_xlabel('Mean Income Rank at Age 35  (kfr_pooled_pooled_p25)', fontsize=12)
ax.set_ylabel('Number of Tracts', fontsize=12)
ax.set_title('Distribution of Mobility Outcomes Across All NYC Census Tracts', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('hc06_nyc_histogram.png', dpi=150)
plt.show()

print(f"NYC mean income rank: {nyc_mean:.4f}")
print(f"This means: the average child from a low-income family in NYC")
print(f"ends up at roughly the {nyc_mean*100:.0f}th percentile nationally.")

### Comparing boroughs

Do some boroughs have systematically higher mobility than others?
An **overlapping histogram** lets us see all five distributions at once,
using a different color for each borough.

Look for: Do the distributions overlap a lot, or are some boroughs clearly separated?
Which borough has the widest spread?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for boro, group in nyc.groupby('borough'):
    vals = group['kfr_pooled_pooled_p25'].dropna()
    ax.hist(vals, bins=25, alpha=0.5, label=f'{boro} (n={len(vals)})',
            color=COLORS[boro], edgecolor='none')

ax.set_xlabel('Mean Income Rank at Age 35  (kfr_pooled_pooled_p25)', fontsize=12)
ax.set_ylabel('Number of Tracts', fontsize=12)
ax.set_title('Mobility Outcomes by Borough', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('hc06_borough_histograms.png', dpi=150)
plt.show()

**A side-by-side bar chart** compares one summary number per group. In the left panel we plot the *mean* for each borough and the right panel the *spread* (standard deviation), one statistic per panel.

This is a different kind of bar chart from the histogram above, and the difference matters. In the histogram, each bar's height was a *count*, how many tracts fell into a bin, so a single borough was made of many bars. Here, each bar's height *is* the statistic we computed, so an entire distribution collapses down to one bar. That is exactly what lets us rank the boroughs at a glance, which the overlapping histogram could not do cleanly.

In the overlapping histogram the mean was a position along the x-axis, not the height of any bar, which made the boroughs hard to compare. Here the mean *is* the bar height, so ranking them is immediate.

In [ ]:
# You have used groupedData['column'].mean() before to get one statistic per group.
# .agg() does the same kind of thing, but lets you ask for several statistics at once:
# here we get both the mean and the standard deviation (std) in a single table.
boro_stats = nyc.groupby('borough')['kfr_pooled_pooled_p25'].agg(['mean', 'std']).round(4)
print("Borough-level statistics for kfr_pooled_pooled_p25:")
print(boro_stats.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: mean income rank per borough
axes[0].bar(boro_stats.index, boro_stats['mean'],
            color=[COLORS[b] for b in boro_stats.index], edgecolor='white')
axes[0].set_ylabel('Mean Income Rank', fontsize=12)
axes[0].set_title('Average Mobility Score by Borough', fontsize=12)
axes[0].set_ylim(0, 0.6)
axes[0].axhline(nyc_mean, color='black', linewidth=1.5, linestyle='--', label=f'NYC mean: {nyc_mean:.3f}')
axes[0].legend(fontsize=10)

# Right: standard deviation (spread) per borough
axes[1].bar(boro_stats.index, boro_stats['std'],
            color=[COLORS[b] for b in boro_stats.index], edgecolor='white')
axes[1].set_ylabel('Standard Deviation', fontsize=12)
axes[1].set_title('Spread of Mobility Scores by Borough', fontsize=12)

for ax in axes:
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('hc06_borough_comparison.png', dpi=150)
plt.show()

### A first map: where are the tracts?

We can use the latitude and longitude from the centroid file to make a simple geographic plot.
Each dot below is one Census tract, placed at its center.
The **color** encodes the mobility score: green = higher opportunity, red = lower.

This is a preview of the interactive Plotly maps you will build in HC 9.

In [ ]:
valid = nyc.dropna(subset=['kfr_pooled_pooled_p25', 'LATITUDE', 'LONGITUDE'])

fig, ax = plt.subplots(figsize=(7, 9))
# ax.scatter draws the dots AND returns an object representing them, which we save as sc.
# sc holds the value-to-color mapping (which kfr value became which color), and the
# colorbar below reads that mapping out of sc to draw and label its color scale.
sc = ax.scatter(
    valid['LONGITUDE'], valid['LATITUDE'],
    c=valid['kfr_pooled_pooled_p25'],
    cmap='RdYlGn',
    # Marker styling: s is the dot size, alpha=0.85 makes dots slightly see-through
    # so overlapping tracts blend instead of hiding each other, and linewidths=0
    # removes the thin outline around each dot for a cleaner look.
    s=12, alpha=0.85, linewidths=0
)
cbar = plt.colorbar(sc, ax=ax, label='Mean Income Rank  (kfr_pooled_pooled_p25)', shrink=0.6)

ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('Economic Mobility Across NYC Census Tracts', fontsize=13)
ax.text(0.5, -0.12, 'green = higher opportunity  |  red = lower opportunity',
        transform=ax.transAxes, ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('hc06_nyc_map.png', dpi=150, bbox_inches='tight')
plt.show()

Even in this simple static map you can start to see patterns.
We will explore these patterns in much more detail in HC 9 with an interactive map where
you can hover over individual tracts and zoom into your neighborhood.

---
## YOUR TURN

Now that the shared NYC dataset is ready, let's look at your specific borough and neighborhood.

### Step 1 — Filter to your borough

**Change `MY_COUNTY` to your team's borough FIPS code.**

In [ ]:
# ── CHANGE THIS to your team's borough ───────────────────────────────────────
# 5=Bronx  47=Brooklyn  61=Manhattan  81=Queens  85=Staten Island
MY_COUNTY       = 61             # replace with your team's borough FIPS code
MY_BOROUGH_NAME = 'Manhattan'    # update to match
# ─────────────────────────────────────────────────────────────────────────────

my_borough = nyc[nyc['county'] == MY_COUNTY].copy()

# TODO — describe your borough. Write the code below.
#
# 1. Report how many tracts your borough has and how many are missing the outcome.
#    Hint: in Part 4 we counted missing values with .isna().sum(), and len(...)
#    gives the total number of rows.
#
# 2. Print summary statistics for your borough's kfr_pooled_pooled_p25 column.
#    Hint: in Part 5 we called .describe().round(4) on the NYC column. Do the
#    same, but on my_borough instead of nyc.
#
# 3. Compute your borough's mean and store it as  boro_mean  — you will reuse it
#    in the next cell. Hint: we used .mean() on a column in Part 5.


### Step 2 — Find your neighborhood's tracts

Use the tract IDs you identified in HC 2 using the NTA crosswalk.
**Update `MY_NEIGHBORHOOD` and `MY_TRACTS`.**

In [ ]:
# ── CHANGE THESE to your neighborhood ────────────────────────────────────────
MY_NEIGHBORHOOD = 'East Harlem'
MY_TRACTS       = [16200, 16400, 17000]   # from your HC 2 NTA crosswalk
# Enter tracts in the SAME format the dataset uses: look at the `tract` column
# printed earlier (e.g. 200, 1600, 16200), not the long GEOID from the crosswalk.
# ─────────────────────────────────────────────────────────────────────────────

my_nbhd = my_borough[my_borough['tract'].isin(MY_TRACTS)]

# TODO — take a closer look at your neighborhood. Write the code below.
#
# 1. Display your neighborhood's tracts with their outcome values.
#    Hint: pick the columns you care about (tract, kfr_pooled_pooled_p25, ...)
#    and print them, the way we previewed rows with .head() earlier.
#
# 2. Compute your neighborhood's mean kfr_pooled_pooled_p25 and compare it to
#    boro_mean and nyc_mean. Is your neighborhood above or below each?
#    Hint: .mean() on the column, then compare the numbers.
#
# 3. Make a plot comparing your neighborhood to your borough.
#    Hint: in Part 5 we drew histograms of kfr_pooled_pooled_p25 and marked a
#    mean with ax.axvline(...), and we compared means with a bar chart. Reuse one
#    of those, for example a borough histogram with a vertical line at your
#    neighborhood's mean, or a bar per neighborhood tract's value.


### Step 3 — Plan for HC 7

HC 7 asks your team to produce a full data analysis of your borough. The group component is an
overall look at your borough distribution and a comparison to the other four boroughs (at least six
plots across at least three plot types), and the individual component is each person digging deeper
into their own neighborhood's tracts.

You have already used three plot types in this notebook that you can build on: a **histogram**
(`ax.hist`) for the shape of one distribution, a **bar chart** (`ax.bar`) for comparing a single
number across groups, and a **scatter plot** (`ax.scatter`) for the relationship between two
variables or for placing tracts on a map. A few others worth trying are a **line plot** (`ax.plot`)
for how a value changes across an ordered sequence, a **box plot** (`ax.boxplot`) for comparing the
spread and median of several groups side by side, and a **horizontal bar chart** (`ax.barh`) for
ranking many neighborhoods or tracts by their value, where long names sit neatly on the y-axis and
sorted bars make the ranking easy to read. Pick plot types that fit your question: distributions call
for histograms or box plots, comparisons across boroughs or tracts call for bar charts, and geographic
patterns call for scatter maps.

Discuss with your team and fill in the plan below.


**Your team's HC 7 plan** (fill in each field):

**Team:**

**Borough:**

**Neighborhoods (one per team member):**

**Three questions we will investigate in HC 7:**
1.
2.
3.

**Plot types we plan to use, and why each one fits the question:**
1.
2.
3.

**One thing in this notebook that already surprised us:**


---
## Reusing your data in future assignments 

Later assignment will start from the `nyc_opportunity.csv` you just downloaded to your
computer. Colab erases saved and uploaded files when a session ends, so whenever you use
Colab for a later assignment, at the **start of each new session** you will upload this file back in. 
The cell below shows how. You do not need to run it now, it is here as a template for when you need it.


In [ ]:
# ── RUN THIS AT THE START OF HC 7-10 (you do not need it now) ─────────────────
# Re-uploads the nyc_opportunity.csv you saved to your computer back into Colab.
from google.colab import files
uploaded = files.upload()          # in the file picker, choose your nyc_opportunity.csv

import pandas as pd
nyc = pd.read_csv('nyc_opportunity.csv')   # works as long as the file keeps its name
print('Loaded', len(nyc), 'rows from nyc_opportunity.csv')